In [1]:
# ================================================
# initialize
# ================================================
include("setup_notebook.jl")


MECH3620 Loading modules...
[OK] PyCall: C:/Users/kychandv/miniconda3/envs/mech3620/python.exe
TA_code path: c:\Users\kychandv\MECH3620\3620_project\MECH3620_Project\TA_code
TA_code exists: true

Importing functions...
[OK] Imported mech3620_models module
Available in mech3620_models:
[ERROR] Cannot import mech3620_models: MethodError(names, (PyObject <module 'mech3620_models' from 'c:\\Users\\kychandv\\MECH3620\\3620_project\\MECH3620_Project\\TA_code\\mech3620_models.py'>,), 0x00000000000097df)
[OK] calc_thrust_lapse loaded
[OK] US_Standard_1976_Atmosphere loaded
[OK] calculate_time_to_climb loaded
[OK] calc_TOP_given_BFL_requirement loaded
[OK] Atmosphere instance created

Initialization complete!

Available functions/variables:
  ✓ calc_thrust_lapse
  ✓ US_Standard_1976_Atmosphere (class)
  ✓ atmosphere (instance)
  ✓ calc_TOP


In [ ]:
using PyCall, DataFrames, Plots, LaTeXStrings, StatsPlots, CSV, Dates, CategoricalArrays, Interpolations


In [ ]:
# #Propulsion model: Thrust lapse rate
# h_35000ft = 35000 * 0.3048  # 轉換為米
# thrust_coeff = calc_thrust_lapse(h_35000ft, 250.0, 0.0, 1.0)
# println("推力係數 = $(round(thrust_coeff, digits=3))")


# Atmosphere = pyimport("mech3620_models").US_Standard_1976_Atmosphere
# atm = Atmosphere()

# # 計算 35000 ft 的大氣參數
# h_cruise = 35000 * 0.3048
# atm_data = atm.compute_constants(h_cruise, 0.0)

# rho = atm_data["rho"]
# v_sound = atm_data["v_sound"]
# println("密度 = $(round(rho, digits=3)) kg/m³")
# println("音速 = $(round(v_sound, digits=1)) m/s")


In [ ]:
# calc_TOP = pyimport("mech3620_models").calc_TOP_given_BFL_requirement

# TOFL_required = 1800  # m
# n_engines = 2
# TOP_value = calc_TOP(TOFL_required, n_engines)
# println("需要的 TOP = $(round(TOP_value, digits=1)) lbf/ft²")


In [ ]:
#================================================#
# Aircraft Parameters Here 
#================================================#

# -----------------------------
# Preliminary sizing / mission inputs
# -----------------------------
begin

	
	W_cargo = 2004
	W_payload = 70 * 15 + 70 * 90 + W_cargo# kg
	W_crew = 4 * 90 # kg
	WTO_guess = 50000.0  # Initial Guess in kg, 77162lb
	SFC_Cruise = 0.65/3600 # Source: Raymer's Textbook
	SFC_Loiter = 0.50/3600 # Source: Raymer's Textbook
	LD_Max = 16 # ballpark value
	LD_Cruise = 0.866 * LD_Max # Source: Raymer's Textbook
	R = 1250*1000  # m, (Range from HKG to CRK)
	a = 295.1 # m/s, (got from table)
	M = 0.78
	V_Cruise = M * a #(required to > 0.74 M)
	V_Cruise_kmh = V_Cruise * 3.6
	E_loiter = 45*60 # second
	fuel_adjustment_factor = 1.06
	num = 20
	
	Service_ceiling = 43000 * 0.3048 # m
	Cruise_altitude = 35000* 0.3048 #m
	
#---------------------------------------------------------------------------
# CL/CD
#--------------------------------------------------------------------------------

	CL_cruise = 0.5	
	CL_max_cruise = 1.5
	CL_SLmax = 0.9995
	CL_0_SL = 0.0709
	CLmax_takeoff = 2 #just guess, 
	CLmax_landing = 3# just 
	CL_max_climb = 1.8

	CD = 0.013
	CD_M2 = 0.01583
	sigma_cruise = 0.31 # based on 41000 ft altitude etc
	CD0_cruise = 0.019 # from vsp

	g0 = 9.81     # m/s²
	wing_loading = 1000:8000
	


	n = length(wing_loading)

	
	g = 9.81

	e_cruise = 0.85 # just a guess, pls change if you guys have a better source
	e_takeoff = 0.7 # just a guess, pls change if you guys have a better source 
	e_landing = 0.6 # just a guess, pls change if you guys have a better source
	e_0deg = 0.85       # 
    e_5deg = 0.85       #
    e_10deg = 0.75      # 
	AR = 8# guess
#--------------------------------------------------------------------------------------------------
	WF_Warmup = 0.99 # Source: Note
	WF_Taxi = 0.99 # Source: Note
	takeoff1WF = 0.995 # Source: Note
	takeoff2WF = 0.995 # Source: Note
	climb1WF = 0.98 # Source: Note
	climb2WF = 0.98 # Source: Note
	Descent1WF = 0.99 # Source: Note
	Descent2WF = 0.99 # Source: Note
	landing1WF = 0.992 # Source: Note
	landing2WF = 0.992 # Source: Note
#--------------------------------------------------------------------------------------------------
	T_W_climb_required = 0.32
	sigma_sl = 0.95  # ISA+15°C

	rho_sl = 1.225 # standard sea level air density
	V_stall = 62.8 # just a guess, pls change if you guys have a better source
	
	A_raymer = 1.02
	B_raymer = -0.06
#----------------------------------------------------------------------------------------------------
#Takeoff
#--------------------------------------------------------------------------------------------------------
	TOFL_required = 1800 # in m, by requirement

#-----------------------------------------------------------------------------------------------------------
# Landing
#-------------------------------------------------------------------------------------------------------------------

	s_Landing = 1600 # from requirements
	s_a = 305 # from tutorial code
	r_L = 0.6

#----------------------------------------------------------------------------------------------------------------------
#Climbing
#----------------------------------------------------------------------------------------------------------------------------

	CD0 = 0.02 # just a guess, pls change if you guys have a better source
	n_eng = 2 
	k_s_second  = 1.2
	G_second    = 0.024
	delta_CD0_second = 0.010        # takeoff flaps, gear up
	
	
#----------------------------------------------------------------------------------------------------------------------
#Turning
#----------------------------------------------------------------------------------------------------------------------------

	turn_radius = 3000  
	bank_angle = 25 * π/180  

end;


In [ ]:
function get_isa_atmosphere_metric(h_m::Real)
    T_sl = 288.15    # K (Kelvin)
    P_sl = 101325    # Pa (Pascal)
    R_gas = 287.05   # J/(kg·K)
    gamma = 1.4
    g0 = 9.81     # m/s²
    tropopause_m = 11000  # 11,000 m (約 36,089 ft)

    if h_m < tropopause_m
        T = T_sl - 0.0065 * h_m  # 溫度遞減率 6.5 K/km
        P = P_sl * (T / T_sl)^5.2561
    else
        T_trop = T_sl - 0.0065 * tropopause_m
        P_trop = P_sl * (T_trop / T_sl)^5.2561
        T = T_trop
        P = P_trop * exp(-g0 / (R_gas * T) * (h_m - tropopause_m))
    end

    rho = P / (R_gas * T)    # kg/m³
    a = sqrt(gamma * R_gas * T)  # m/s
    return rho, a
end

function fuel_weight_fraction(final_beta, a)
    Wf_WTO = fuel_adjustment_factor * (1 - final_beta)
    return Wf_WTO
end;

function empty_weight_raymer(WTO, A, B)
	We_WTO = A * (WTO * kg_to_lb)^B
	return We_WTO
end;


In [ ]:
#================================================#
# Preliminary Weight Estimation 
#================================================#


function maximum_takeoff_weight(W_payload, W_crew, Wf_WTO, We_WTO)
	WTO = (W_payload + W_crew) / (1 - Wf_WTO - We_WTO)
	return WTO
end


function cruise_weight_fraction(R, SFC, V, L_D)
	cruiseWF = exp(-R * SFC / (V * L_D))
	return cruiseWF
end;

cruise1WF = cruise_weight_fraction(R, SFC_Cruise, V_Cruise, LD_Cruise)
cruise2WF = cruise_weight_fraction(R, SFC_Cruise, V_Cruise, LD_Cruise)



function loiter_weight_fraction(E_loiter, SFC, L_D)
	loiterWF = exp(-E_loiter * SFC / L_D)
	return loiterWF
end;

loiter1WF = loiter_weight_fraction(E_loiter, SFC_Loiter, LD_Max)
loiter2WF = loiter_weight_fraction(E_loiter, SFC_Loiter, LD_Max)

warmup1 = WF_Warmup
taxi1 = WF_Taxi
warmup2 = WF_Warmup
taxi2 = WF_Taxi


WFs = [warmup1, taxi1, takeoff1WF, climb1WF, cruise1WF, loiter1WF, Descent1WF, landing1WF]
WFs_return = [warmup2, taxi2, takeoff2WF, climb2WF, cruise2WF, loiter2WF, Descent2WF, landing2WF]

beta_mission = cumprod(WFs)

begin
    beta_warmup1 = beta_mission[1]
    beta_taxi1 = beta_mission[2]
	beta_takeoff1 = beta_mission[3]
	beta_climb1 = beta_mission[4]
	beta_cruise1 = beta_mission[5]
	beta_loiter1 = beta_mission[6]
    beta_descent1 = beta_mission[7]
	beta_landing1 = beta_mission[8]
end;



In [ ]:


println("="^60)
println("MISSION ANALYSIS RESULTS")
println("="^60)

println("\n WEIGHT FRACTIONS (WF):")
println("-"^60)
println("warmup1 = ", warmup1)        # 0.99
println("taxi1 = ", taxi1)            # 0.99  
println("takeoff1WF = ", takeoff1WF)  # 0.995
println("climb1WF = ", climb1WF)      # 0.98
println("cruise1WF = ", cruise1WF)    # 0.94702
println("loiter1WF = ", loiter1WF)    # 0.99999
println("Descent1WF = ", Descent1WF)  # 0.99
println("landing1WF = ", landing1WF)  # 0.992
println("-"^60)
println("Product of all WFs = ", prod(WFs))
println("="^60)

println("\n📈 BETA VALUES - product of fuel weight ratio:")
println("-"^60)
println("beta_warmup1 = ", beta_warmup1)    # 0.99
println("beta_taxi1 = ", beta_taxi1)        # 0.9801
println("beta_takeoff1 = ", beta_takeoff1)  # 0.9752
println("beta_climb1 = ", beta_climb1)      # 0.9557
println("beta_cruise1 = ", beta_cruise1)    # 0.9051
println("beta_loiter1 = ", beta_loiter1)    # 0.9051
println("beta_descent1 = ", beta_descent1)  # 0.8960
println("beta_landing1 = ", beta_landing1)  # 0.8888
println("-"^60)
println("Total fuel burn = $(round((1-beta_landing1)*100, digits=1))%")
println("="^60)

#For return-------------------------------------------------------------------

begin
    println("\n" * "="^60)
    println("return BETA calculation")
    println("="^60)
    
    
    correct_start_return = beta_landing1
    return_WFs = [warmup2, taxi2, takeoff2WF, climb2WF, cruise1WF, loiter1WF, Descent2WF, landing2WF]
    return_beta_value = Float64[]


    println("\n3. Return Segment WF:")
    beta = correct_start_return
    phases = ["Warmup", "Taxi", "Takeoff", "Climb", "Cruise", "Loiter", "Descent", "Landing"]
    
    for (i, (phase, wf)) in enumerate(zip(phases, return_WFs))
        beta *= wf
        push!(return_beta_value, beta)
        println("   $phase: WF=$(round(wf, digits=4)), Beta=$(round(beta, digits=6))")
    end
    
    
    println("\n4. Comparison of Fuel Consumed:")
    outbound_burn = 1 - beta_landing1
    return_burn = 1 - (beta / correct_start_return)
    
    println("   Outboard Fuel Consumption: $(round(outbound_burn*100, digits=1))%")
    println("   Return Fuel Consumption: $(round(return_burn*100, digits=1))%")
    
    if abs(return_burn - outbound_burn) < 0.05
        println("   ✅ The fuel consumed are similar, make sense")
    else
        println("   ⚠️ return $(round(return_beta_value[end]*100, digits=1))% didnt match outboard $(round(outbound_burn*100, digits=1))% ")
        println("   check plz (cruise2WF)")
    end
    println("="^60)

    global final_beta = return_beta_value[end]
end

Wf_WTO = fuel_weight_fraction(final_beta, fuel_adjustment_factor)
println("="^60)
println("Mission fuel fraction (without trapped): $(round((1-final_beta)*100, digits=4))%")
println("Total fuel fraction (with 6% trapped): $(round((Wf_WTO)*100, digits=4))%")


In [ ]:
#---------------------------------------------------------------------------------
# Raymer's Model 
#------------------------------------------------------------------------------------

# function empty_weight_raymer(WTO, A, B)
# 	We_WTO = A * (WTO * kg_to_lb)^B
# 	return We_WTO


# println("\nRAYMER'S MODEL:")
# println("-"^40)
# println("  MTOW = $(round(WTO_final, digits=1)) kg")
# println("  Empty weight fraction = $(round(We_WTO_final, digits=4))") 
# println("  Empty weight = $(round(We_raymer, digits=0)) kg")  

#---------------------------------------------------------------------------------
# Roskam's Model (need to fix something)
#------------------------------------------------------------------------------------
# A_roskam = 0.08
# B_roskam = 1.04

#    # Roskam's eq: log10(WTO) = A + B × log10(We)
#     # so We = 10^((log10(WTO) - A) / B)
#     log10_We = (log10(WTO_final_lb) - A_roskam) / B_roskam
#     We_roskam_lb = 10^log10_We
#     We_WTO_roskam = We_roskam_lb / WTO_guess_lb
#     println("\n" * "="^60)
#     println("ROSKAM'S EMPTY WEIGHT MODEL")
#     println("="^60)

#     We_roskam = We_roskam_lb / kg_to_lb

#     println("\nRoskam's Model Results:")
#     println("  MTOW = $(round(WTO_final, digits=1)) kg")
#     println("  Empty Weight = $(round(We_roskam, digits=1)) kg")
#     println("  Empty Weight Fraction = $(round(We_WTO_roskam, digits=4))")


In [ ]:
#================================================#
# ITERATIVE WEIGHT SIZING LOOP
#================================================#

begin
    
    println("\n" * "="^60)
    println("ITERATIVE WEIGHT SIZING")
    println("="^60)

    # 迭代參數
    WTO_old = WTO_guess  # 34000 kg
    max_iter = 20
    tolerance = 0.00005  
    converged = false
    iteration = 1
    WTO_new = WTO_old
    W_pl_crew = W_payload + W_crew  # 7350 + 360 = 7710 kg
    # unit convertion
    kg_to_lb = 2.20462
    lb_to_kg = 1 / kg_to_lb
    
    # 固定燃油分數（從你之前的計算）
    
    # 儲存迭代歷史
    history = DataFrame(
        Iteration = Int[],
        WTO_kg = Float64[],
        We_WTO_raymer = Float64[],
        We_raymer = Float64[],
        Wf_WTO = Float64[],
        Error = Float64[]
    )

    while !converged && iteration ≤ max_iter
        
        println("\n" * "-"^60)
        println("迭代 #$iteration")
        println("-"^60)
        
        WTO_old_lb = WTO_old * kg_to_lb
        We_WTO_raymer = A_raymer * (WTO_old_lb)^B_raymer
        We_raymer_lb = We_WTO_raymer * WTO_old_lb
        We_raymer = We_raymer_lb * lb_to_kg  # 轉回公斤
        W_pl_crew = W_payload + W_crew  # 7350 + 360 = 7710 kg
        WTO_new = (W_pl_crew) / (1 - Wf_WTO - We_WTO_raymer)
        error = abs(WTO_new - WTO_old) / WTO_old
        
        push!(history, (
            iteration, 
            round(WTO_old, digits=1), 
            round(We_WTO_raymer, digits=4), 
            round(We_raymer, digits=1), 
            round(Wf_WTO, digits=4), 
            round(error*100, digits=2)
        ))
        
        println("  WTO_old = $(round(WTO_old, digits=1)) kg")
        println("  WTO_old_lb = $(round(WTO_old_lb, digits=1)) lb")
        println("  We_WTO = $(round(We_WTO_raymer, digits=4))")
        println("  We = $(round(We_raymer, digits=1)) kg")
        println("  Wf_WTO = $(round(Wf_WTO, digits=4))")
        println("  W_pl_crew = $(round(W_pl_crew, digits=1)) kg")
        println("  WTO_new = $(round(WTO_new, digits=1)) kg")
        println("  Error = $(round(error*100, digits=2))%")
        
        # 9. 檢查收斂
        if error < tolerance
            converged = true
            println("\n✅ 收斂！最終 MTOW = $(round(WTO_new, digits=1)) kg")
        else
            # 使用鬆弛因子更新 WTO_old
            relaxation = 0.5
            WTO_old = (1 - relaxation) * WTO_old + relaxation * WTO_new
            iteration += 1
        end
    end

    # 顯示迭代歷史
    println("\n" * "="^60)
    println("迭代歷史總結")
    println("="^60)
    println(history)

    # 最終結果
    WTO_final = WTO_new
    # 注意：這裡要用最後一次的 We_WTO_raymer
    
    We_WTO_final = history[end, "We_WTO_raymer"]
    We_final = We_WTO_final * WTO_final  # We_WTO_raymer 是分數，可以直接乘
    Wf_final = Wf_WTO * WTO_final
    
    println("\n" * "="^60)
    println("Final Design weight (After Iteration)")
    println("="^60)
    println("MTOW:        $(round(WTO_final, digits=1)) kg")
    println("OEW:         $(round(We_final, digits=1)) kg")
    println("燃油重量:    $(round(Wf_final, digits=1)) kg")
    println("載重+機組:   $(round(W_pl_crew, digits=1)) kg")

    # 檢查重量平衡
    weight_sum = We_final + Wf_final + W_pl_crew
    balance_error = (weight_sum - WTO_final) / WTO_final * 100
    println("\n重量平衡檢查:")
    println("  We + Wf + W_pl_crew = $(round(weight_sum, digits=1)) kg")
    println("  MTOW = $(round(WTO_final, digits=1)) kg")
    println("  誤差 = $(round(balance_error, digits=2))%")
    if abs(balance_error) < 0.1
        println("  ✅ 平衡")
    else
        println("  ⚠️ 不平衡，檢查計算")
    end
    
end

#-----------------------------------------------------------------------------------------
# Plotting Here
#-----------------------------------------------------------------------------------------
begin
    
    WTO_list = history[:, "WTO_kg"]  # Get all rows from WTO_kg column
    error_list_plot = history[:, "Error"]  # Get all rows from Error column
    
    println("="^60)
    println("ITERATION CONVERGENCE PLOT")
    println("="^60)
    println("Initial MTOW: $(round(WTO_list[1], digits=1)) kg")
    println("Final MTOW:   $(round(WTO_list[end], digits=1)) kg")
    println("Iterations:   $(length(WTO_list)-1)")
    println("Final error:  $(round(error_list_plot[end], digits=6))")
    println("="^60)
    
    # Plotting
    layout = @layout [a{0.6h}; b{0.4h}]
    
   
    p1 = plot(0:length(WTO_list)-1, WTO_list,
              label = "MTOW",
              ylabel = "MTOW (kg)", 
              xlabel = "",
              linewidth = 2,
              color = :blue,
              marker = :circle,
              grid = true,
              legend = :topright)
    
    
    p2 = plot(1:length(error_list_plot), error_list_plot,
              label = "Error",
              ylabel = "Error", 
              xlabel = "Iteration",
              yscale = :log10,
              linewidth = 2,
              color = :red,
              marker = :square,
              grid = true,
              legend = :topright)
    
    # combine the plotting
    p_final = plot(p1, p2, layout = layout, size=(700, 700),
                   title = "Fixed-point Iteration Convergence")
    
    display(p_final)
    savefig(p_final, "iteration_convergence.pdf")
    println("✅ Plot saved")
end


In [ ]:
#================================================#
# Drag Polar
#================================================#
#-----------------------------------------------------------------------------------------#

function induced_drag_coefficient(e, AR)
	K = 1 / (pi * e * AR)
	return K
end;

begin
	K_cruise = induced_drag_coefficient(e_cruise, AR)
	K_takeoff = induced_drag_coefficient(e_takeoff, AR)
	K_landing = induced_drag_coefficient(e_landing, AR)
end;

function drag_polar(CD0, K, CL, CL0 = 0.)
	CD = CD0 + K * (CL - CL0)^2
	return CD
end;


In [ ]:
#Mission Profile


#=================================================================================#
# Takeoff/ Taxi
#=================================================================================#

#-----------------------------------------------------------------------
function dynamic_pressure(rho, V)
	q = 0.5 * rho * V^2
	return q
end;

TOP_required = TOFL_required * 3.28084 / 37.5

function takeoff_condition(WbS, sigma, CL_takeoff, TOP)
	TbW_takeoff = (0.0929/4.448) * WbS / (sigma * CL_takeoff * TOP)
	return TbW_takeoff
end;

TbW_takeoff = takeoff_condition.(wing_loading, sigma_sl, CLmax_takeoff, TOP_required)


In [ ]:
#=================================================================================#
# Climb
#=================================================================================#


	#----------------------------------------------------------------
function climb_condition(k_s, CD0, CL_max, K, G)
	TbW_climb = k_s^2 * CD0 / CL_max + K * CL_max / k_s^2 + G
	return TbW_climb
end;

function thrust_corrected_climb(k_s, CD0, CL_max, K, G, n_eng, # Necessary inputs
	weight_factor = 1; 		 # Optional input with default=1
	MCT = false, OEI = false # Named arguments	
	)

	if OEI # one-engine-inoperative scenario
		OEI_factor = n_eng / (n_eng - 1)
	else
		OEI_factor = 1
	end

	if MCT # maximum continuous thrust condition
		MCT_factor = 1 / 0.94
	else
		MCT_factor = 1
	end

	TbW_climb_corrected = (1/0.8) * MCT_factor * OEI_factor * weight_factor * climb_condition(k_s, CD0, CL_max, K, G)

	return TbW_climb_corrected
end;
#####################################################################################
## Takeoff Climb OEI (Climb 1)
####################################################################################
begin	
	k_s_takeoff  = 1.2
	G_takeoff 	 = 0.012
	delta_CD0_takeoff = 0.035 # Flaps 5 deg, gear down
	K_takeoff 	 = induced_drag_coefficient(e_5deg, AR)
	WF_takeoff_climb = 0.99 * beta_takeoff1 # Takeoff climb weight fraction
	
	TbW_takeoff_climb = thrust_corrected_climb(k_s_takeoff, CD0 + delta_CD0_takeoff, CL_max_climb, K_takeoff, G_takeoff, n_eng, WF_takeoff_climb, OEI = false)
	
	takeoff_climbs = fill(TbW_takeoff_climb, length(wing_loading))
end;

#####################################################################################
## Transition Climb OEI (Climb 2)
####################################################################################
begin	
	k_s_trans 	= 1.1
	G_trans 	= 0
	delta_CD0_trans 	= 0.030 # Flaps 10 deg, gear up
	K_climb 	= induced_drag_coefficient(e_10deg, AR)
	beta_trans 	= 0.99 * beta_climb1 # Weight fraction at transition climb 
	
	TbW_trans_climb = thrust_corrected_climb(k_s_trans, CD0 + delta_CD0_trans, CL_max_climb, K_climb, G_trans, n_eng, beta_trans, OEI = true)
	
	trans_climb = fill(TbW_trans_climb, length(wing_loading))
end;


#####################################################################################
## Second Climb OEI (Climb 3)
####################################################################################
begin	
	k_s_second 	= 1.2
	G_second 	= 0.024
	delta_CD0_second = 0.01 # Flaps 5 deg, gear up
	K_second 	= induced_drag_coefficient(e_5deg, AR)
	beta_second   = beta_takeoff1 # Weight fraction at second climb

	TbW_second_climb = thrust_corrected_climb(k_s_second, CD0 + delta_CD0_second, CL_max_climb, K_second, G_second, n_eng, beta_second, OEI = true)
	
	second_climb = fill(TbW_second_climb, length(wing_loading))
end;
#####################################################################################
## Enroute Climb OEI (Climb 4)
####################################################################################
begin	
	k_s_enroute  = 1.25
	G_enroute 	 = 0.012
	delta_CD0_enroute = 0.0 # Clean, i.e. no flaps
	K_enroute 	 = induced_drag_coefficient(e_0deg, AR)
	beta_enroute 	 = 0.98 * beta_climb1# Weight fraction at enroute climb

	TbW_enroute_climb = thrust_corrected_climb(k_s_enroute, CD0 + delta_CD0_enroute, CL_max_climb, K_enroute, G_enroute, n_eng, beta_enroute, MCT = true, OEI = true)
	
	enroute_climb = fill(TbW_enroute_climb, length(wing_loading))
end;

#####################################################################################
## Balked Landing Climb AEO (Climb 5)
####################################################################################
begin	
	k_s_balked_AEO  = 1.3
	G_balked_AEO    = 0.032
	delta_CD0_balked_AEO = 0.030 # Flaps 10 deg, gear up
	K_balked_AEO    = induced_drag_coefficient(e_10deg, AR)
	beta_balked_AEO   = beta_landing1 # NEED TO CHECK

	TbW_balked_AEO = thrust_corrected_climb(k_s_balked_AEO, CD0 + delta_CD0_balked_AEO, CL_max_climb, K_balked_AEO, G_balked_AEO, n_eng, beta_balked_AEO)
	
	balked_AEO_climb = fill(TbW_balked_AEO, length(wing_loading))
end;


#####################################################################################
## Balked Landing Climb OEI (Climb 6)
####################################################################################
begin	
	k_s_balked_OEI = 1.5
	G_balked_OEI = 0.021
	delta_CD0_balked_OEI = 0.030 # Flaps 10 deg, gear up
	K_balked_OEI = induced_drag_coefficient(e_10deg, AR)
	beta_balked_OEI = beta_landing1 # Maximum landing weight fraction

	TbW_balked_OEI = thrust_corrected_climb(k_s_balked_OEI, CD0 + delta_CD0_balked_OEI, CL_max_climb, K_balked_OEI, G_balked_OEI, n_eng, beta_balked_OEI, OEI = true)
	
	balked_OEI_climb = fill(TbW_balked_OEI, length(wing_loading))
end;




In [ ]:
#=================================================================================#
# Cruise
#=================================================================================#

#--------------------------------------------------------------------#
function cruise_condition(wing_loading, q, CD0_cruise, K)
	TbW_cruise = q * CD0_cruise / wing_loading + K / q * wing_loading
	return TbW_cruise
end;

begin
    rho_cruise, a_cruise = get_isa_atmosphere_metric(Cruise_altitude)
    sigma_cruise = rho_cruise / rho_sl
    q = dynamic_pressure(rho_cruise, M * a_cruise)
    TbW_cruise = 1 / sigma_cruise^0.6 * beta_cruise1 * cruise_condition.(wing_loading, q, CD0_cruise, K_cruise)
end

begin
    ROC_ceiling = 0.508
    G_ceiling = ROC_ceiling / V_Cruise
    rho_ceiling, a_ceiling = get_isa_atmosphere_metric(Service_ceiling)
    q_ceiling = dynamic_pressure(rho_ceiling, V_Cruise)
    sigma_ceiling = rho_ceiling / rho_sl

    # 计算每个翼载对应的升力系数
    CL_ceiling = wing_loading ./ q_ceiling   # 注意使用点除

    # 广播调用 thrust_corrected_climb，对每个翼载使用对应的 CL_ceiling
    TbW_ceiling = thrust_corrected_climb.(1.0, CD0_cruise, CL_ceiling, K_cruise, G_ceiling, n_eng, beta_cruise1)
    # 注意：需要在函数调用后加上推力衰减修正
    TbW_ceiling = (1 / sigma_ceiling^0.6) .* TbW_ceiling
end


In [ ]:
#================================================#
# Turning
#================================================#
function turn_condition(W_S, V, R, phi, CD0, K)
    q = 0.5 * rho_sl * sigma_cruise * V^2
    
    n = 1 / cos(phi)

    CL_turn = n * W_S / q
    
    CD_turn = CD0 + K * CL_turn^2
    T_W = CD_turn * q / W_S
    
    return T_W
end
begin
    TbW_turn = turn_condition.(wing_loading, V_Cruise, turn_radius, bank_angle, CD0_cruise, K_cruise)
end


In [ ]:


#=================================================================================#
# Landing/ Taxi
#=================================================================================#



#----------------------------------------------------------------------
function landing_wing_loading(sigma, CLmax_landing, s_FL, s_a, r, g)
	WbS_landing = sigma * g * CLmax_landing / 5 * (r * s_FL - s_a)
	return WbS_landing  
end;


begin
	WbS_landing_max = 1/beta_landing1 * landing_wing_loading(sigma_sl, CLmax_landing, s_Landing, s_a, r_L, g0)
	println("Landing maximum W/S = $(round(WbS_landing_max, digits=0)) N/m²")
	WbS_landing = fill(WbS_landing_max, n)
end;

begin
	WbS_landing_max2 = 1/beta * landing_wing_loading(sigma_sl, CLmax_landing, s_Landing, s_a, r_L, g0)
	println("Landing maximum W/S = $(round(WbS_landing_max2, digits=0)) N/m²")
	WbS_landing2 = fill(WbS_landing_max2, n)
end;



In [ ]:

#================================================#
# Acceleration
#================================================#

Target_Air_speed = V_Cruise # m/s, just a guess, pls change if you guys have a better source
V1_takeoff = 1.13 * V_stall       # 起飞安全速度 (V2) ≈ 1.13 V_stall
V2_takeoff = 1.25 * V_stall       # 初始爬升速度 (V_climb) ≈ 1.25 V_stall
dt_takeoff = 30              # 加速时间 (秒) - FAR 25要求的30秒
CD0_takeoff = 0.035 
beta_takeoff_accel = beta_takeoff1
rho_takeoff = rho_sl * sigma_sl
M_climb_top = 0.5                      # 爬升顶点马赫数
h_climb_top = 9000                      # (m) ≈ 30000ft



rho_climb_top, a_climb_top = get_isa_atmosphere_metric(h_climb_top) 
    

V1_cruise = M_climb_top * a_climb_top   # 爬升顶点速度
V2_cruise = V_Cruise                    # 巡航速度
dt_cruise = 120                          # 加速时间 (秒) - 2分钟
CD0_cruise_accel = CD0_cruise            # 巡航构型阻力
beta_cruise_accel = beta_cruise1          # 巡航段权重系数
rho_cruise_accel = rho_climb_top

function acceleration_condition(W_S, V1, V2, dt, CD0, K, beta, rho)
    
    V_avg = (V1 + V2) / 2
    q = 0.5 * rho * V_avg^2
    
    CL_accel = W_S / q
    CD_accel = CD0 + K * CL_accel^2

    acceleration_term = (V2 - V1) / (g * dt)
    drag_term = q * CD0 / W_S + (K / q) * W_S
    
    T_W_accel = (acceleration_term + drag_term) / beta
    return T_W_accel

end
begin
     TbW_takeoff_accel = acceleration_condition.(
        wing_loading,
        V1_takeoff,
        V2_takeoff,
        dt_takeoff,
        CD0_takeoff,
        K_takeoff,
        beta_takeoff_accel,
        rho_takeoff
    ) 
end

begin
    
    TbW_cruise_accel = acceleration_condition.(
        wing_loading,
        V1_cruise,
        V2_cruise,
        dt_cruise,
        CD0_cruise_accel,
        K_cruise,
        beta_cruise_accel,
        rho_cruise_accel
    )
end




In [ ]:
#================================================#
# Loiter
#================================================#

V_loiter = 110            # in m/s
h_loiter = 15000 * 0.3048         # in m

rho_loiter, a_loiter = get_isa_atmosphere_metric(h_loiter)

function loiter_condition(W_S, V, rho, CD0, K, beta)
    
    q = 0.5 * rho * V^2
    
   
    CL_loiter = W_S / q
    
   
    CD_loiter = CD0 + K * CL_loiter^2
    T_W_loiter = CD_loiter * q / W_S
    
    T_W_required = T_W_loiter / beta_loiter1
    
    return T_W_required
end

T_W_loiter = loiter_condition.(wing_loading, 
    V_loiter, rho_loiter, CD0_cruise, K_cruise, beta_loiter1)






In [ ]:
#=================================================================================#
# Landing/ Taxi
#=================================================================================#

#----------------------------------------------------------------------
function landing_wing_loading(sigma, CLmax_landing, s_FL, s_a, r, g)
	WbS_landing = sigma * g * CLmax_landing / 5 * (r * s_FL - s_a)
	return WbS_landing  
end;


begin
	WbS_landing_max = 1/beta_landing1 * landing_wing_loading(sigma_sl, CLmax_landing, s_Landing, s_a, r_L, g0)
	println("Landing maximum W/S = $(round(WbS_landing_max, digits=0)) N/m²")
	WbS_landing = fill(WbS_landing_max, n)
end;

begin
	WbS_landing_max2 = 1/beta * landing_wing_loading(sigma_sl, CLmax_landing, s_Landing, s_a, r_L, g0)
	println("Landing maximum W/S = $(round(WbS_landing_max2, digits=0)) N/m²")
	WbS_landing2 = fill(WbS_landing_max2, n)
end;




In [ ]:
#================================================#
# Stall Speed
#================================================#
  
#-----------------------------------------------------------------------
function dynamic_pressure(rho, V)
	q = 0.5 * rho * V^2
	return q
end;


function wing_loading_stall_speed(V_stall, CL_max_cruise, rho)
	WbS_stall = dynamic_pressure(rho, V_stall) * CL_max_cruise/ beta_cruise1
	return WbS_stall
end;

WbS_stall_Cruise = wing_loading_stall_speed(V_stall, CL_max_cruise, 1.225)

begin
	n = 15
	stalls = fill(WbS_stall_Cruise, n)
	TbWs = range(0, 1, length=n)
end;
println("Stall maximum W/S = $(round(WbS_stall_Cruise, digits=0)) N/m²")


In [ ]:
#================================================#
# Automatic Design Point Selection
#================================================#

begin
    println("="^60)
    println("AUTOMATIC DESIGN POINT SELECTION")
    println("="^60)
    if !@isdefined(WbS_stall_Cruise) || !@isdefined(WbS_landing_max)
        error("Please run previous cells first to define WbS_stall and WbS_landing_max")
    end
    # 1. 创建插值函数（将所有约束曲线转换为函数）
    itp_takeoff = linear_interpolation(wing_loading, TbW_takeoff, extrapolation_bc = Line())
    itp_climb1 = linear_interpolation(wing_loading, takeoff_climbs, extrapolation_bc = Line())
    itp_climb2 = linear_interpolation(wing_loading, trans_climb, extrapolation_bc = Line())
    itp_climb3 = linear_interpolation(wing_loading, second_climb, extrapolation_bc = Line())
    itp_climb4 = linear_interpolation(wing_loading, enroute_climb, extrapolation_bc = Line())
    itp_climb5 = linear_interpolation(wing_loading, balked_AEO_climb, extrapolation_bc = Line())
    itp_climb6 = linear_interpolation(wing_loading, balked_OEI_climb, extrapolation_bc = Line())
    itp_cruise = linear_interpolation(wing_loading, TbW_cruise, extrapolation_bc = Line())
    itp_turn = linear_interpolation(wing_loading, TbW_turn, extrapolation_bc = Line())
    itp_loiter = linear_interpolation(wing_loading, T_W_loiter, extrapolation_bc = Line())
    itp_ceiling = linear_interpolation(wing_loading, TbW_ceiling, extrapolation_bc = Line())
    
   # 2. 确定可行翼载范围
    W_S_max_stall = WbS_stall_Cruise      # 失速限制：W/S ≤ 此值
    W_S_max_landing = WbS_landing_max     # 著陸限制：W/S ≤ 此值（您的代碼中已有 WbS_landing_max）
    W_S_max = min(W_S_max_stall, W_S_max_landing)  # 取兩者較小值作為上限
    W_S_min = 2000  # 合理的最小翼載，避免搜索過低數值

    println("Feasible wing loading range:")
    println("  Stall limit: W/S ≤ $(round(W_S_max_stall, digits=1)) N/m²")
    println("  Landing limit: W/S ≤ $(round(W_S_max_landing, digits=1)) N/m²")
    println("  Combined max: W/S ≤ $(round(W_S_max, digits=1)) N/m²")
        
        println("Feasible wing loading range:")
        println("  W/S <= $(round(W_S_min, digits=1)) N/m² (Stall)")
   
    
    # 3. 在可行范围内搜索设计点
    # 创建更精细的搜索点（在可行范围内取200个点）
    W_S_search = range(max(W_S_min, 2000), min(W_S_max, 8000), length=200)
    
    # 计算每个搜索点所需的最大 T/W
    T_W_required = zeros(length(W_S_search))
    for (i, ws) in enumerate(W_S_search)
        T_W_required[i] = max(
            itp_takeoff(ws),
            itp_climb1(ws),
            itp_climb2(ws),
            itp_climb3(ws),
            itp_climb4(ws),
            itp_climb5(ws),
            itp_climb6(ws),
            itp_cruise(ws),
            itp_turn(ws),
            itp_loiter(ws),
            itp_ceiling(ws)
        )
    end
    
    # 4. 找到最小 T/W 对应的翼载
    idx = argmin(T_W_required)
    W_S_design_auto = W_S_search[idx]
    T_W_design_auto = T_W_required[idx]
    
    println("\n" * "-"^60)
    println("DESIGN POINT (AUTO-SELECTED)")
    println("-"^60)
    println("Wing Loading (W/S): $(round(W_S_design_auto, digits=1)) N/m²")
    println("Thrust-to-Weight (T/W): $(round(T_W_design_auto, digits=3))")
    println("-"^60)
    
    # 5. 验证设计点是否满足所有约束
    println("\nVERIFICATION - Design point vs. each constraint:")
    println("  Takeoff:      $(round(itp_takeoff(W_S_design_auto), digits=3))")
    println("  Climb 1:      $(round(itp_climb1(W_S_design_auto), digits=3))")
    println("  Climb 2:      $(round(itp_climb2(W_S_design_auto), digits=3))")
    println("  Climb 3:      $(round(itp_climb3(W_S_design_auto), digits=3))")
    println("  Climb 4:      $(round(itp_climb4(W_S_design_auto), digits=3))")
    println("  Climb 5:      $(round(itp_climb5(W_S_design_auto), digits=3))")
    println("  Climb 6:      $(round(itp_climb6(W_S_design_auto), digits=3))")
    println("  Cruise:       $(round(itp_cruise(W_S_design_auto), digits=3))")
    println("  Turn:         $(round(itp_turn(W_S_design_auto), digits=3))")
    println("  Loiter:       $(round(itp_loiter(W_S_design_auto), digits=3))")
    println("  Service Ceiling: $(round(itp_ceiling(W_S_design_auto), digits=3))")
    println("  → Required T/W: $(round(T_W_design_auto, digits=3))")
    
    
    # 6. 检查是否在可行范围内
    if W_S_design_auto >= W_S_min && W_S_design_auto <= W_S_max
        println("\n✅ Design point is within stall and landing limits.")
    else
        println("\n❌ Design point is outside stall/landing limits!")
    end
    
    println("="^60)
end


In [ ]:
#-----------------------------------------------------------
# Plotting of Constraint analysis
#-------------------------------------------------------------
begin
    # ✅ 確保所有曲線有正確長度
    n_points = length(wing_loading)
    
    # 檢查並修正長度
    if length(stalls) != n_points
        stalls = fill(WbS_stall_Cruise, n_points)
        TbWs = range(0, 1, length=n_points)
    end
    
    if length(WbS_landing) != n_points
        WbS_landing = fill(WbS_landing_max, n_points)
    end
    
    p_design_point = plot(
        xlabel = "Wing Loading (W/S), N/m²", 
        ylabel = "Thrust Loading (T/W)", 
        title = "Matching Chart",
        legend = :topright,
        legendfontsize = 7,
        grid = true,
        minorgrid = true,
        size = (1000, 750),  
        dpi = 300,
        ylims = (0, 0.6),
        xlims = (850, 8000),
    )
    
    # 繪製所有曲線
    plot!(stalls, TbWs, label = "Stall", color = :gray50)
    plot!(wing_loading, TbW_takeoff, label = "Takeoff", color = :blue)
    plot!(WbS_landing, TbWs, label = "Landing", color = :teal)
    plot!(wing_loading, takeoff_climbs, label = "Takeoff Climb", color = :lightblue)
    plot!(wing_loading, trans_climb, label = "Transition Climb OEI", color = :cyan)
    plot!(wing_loading, second_climb, label = "Second Climb OEI", color = :red)
    plot!(wing_loading, enroute_climb, label = "Enroute Climb OEI", color = :orange)
    plot!(wing_loading, balked_AEO_climb, label = "Balked Landing Climb AEO", color = :violet)
    plot!(wing_loading, balked_OEI_climb, label = "Balked Landing Climb OEI", color = :purple)
    plot!(wing_loading, TbW_cruise, label = "Cruise", color = :green)
    plot!(wing_loading, T_W_loiter, label = "Loiter", color = :indigo)
    plot!(wing_loading, TbW_ceiling, label = "Service Ceiling", color = :magenta)
    plot!(wing_loading, TbW_turn, label = "Turning", color = :brown)
    plot!(wing_loading, TbW_takeoff_accel, label = "Takeoff Acceleration", color = :darkred)
    plot!(wing_loading, TbW_cruise_accel, label = "Cruise Acceleration", color = :black)
    
    display(p_design_point)
end


In [ ]:
#---------------------------------------#
#Find the interestion point here#
#---------------------------------------#
begin
    critical_points = []
    point_names = []


    # 1. Stall 與 Takeoff 交點 (垂直線與曲線)
    W_S_stall = WbS_stall_Cruise;  # 失速限制翼載
    idx_stall = argmin(abs.(wing_loading .- W_S_stall));
    T_W_stall_takeoff = TbW_takeoff[idx_stall];
    push!(critical_points, (W_S_stall, T_W_stall_takeoff));
    push!(point_names, "Stall-Takeoff");
    println("\n1. Stall 與 Takeoff 交點:");
    println("   W/S = $(round(W_S_stall, digits=1)) N/m²");
    println("   T/W = $(round(T_W_stall_takeoff, digits=3))");
    println("W_S_stall,, T_W_stall_takeoff for this intersection point");

    # 2. Landing 與 Takeoff 交點 (新增)
    W_S_landing = WbS_landing_max
    idx_landing = argmin(abs.(wing_loading .- W_S_landing))
    T_W_landing_takeoff = TbW_takeoff[idx_landing]
    push!(critical_points, (W_S_landing, T_W_landing_takeoff))
    push!(point_names, "Landing-Takeoff")
    println("\n2. Landing 與 Takeoff 交點:")
    println("   W/S = $(round(W_S_landing, digits=1)) N/m²")
    println("   T/W = $(round(T_W_landing_takeoff, digits=3))")

	
    # 3. Takeoff 與 Climb 6 交點 (兩條曲線相交)
    diff_takeoff_climb6 = abs.(TbW_takeoff - balked_OEI_climb);
    idx_takeoff_climb6 = argmin(diff_takeoff_climb6);
    W_S_takeoff_climb6 = wing_loading[idx_takeoff_climb6];
    T_W_takeoff_climb6 = TbW_takeoff[idx_takeoff_climb6];  # 或 balked_OEI_climb[idx_takeoff_climb6]
    push!(critical_points, (W_S_takeoff_climb6, T_W_takeoff_climb6));
    push!(point_names, "Takeoff-Climb6");
    println("\n3. Takeoff 與 Climb 6 交點:");
    println("   W/S = $(round(W_S_takeoff_climb6, digits=1)) N/m²");
    println("   T/W = $(round(T_W_takeoff_climb6, digits=3))");
    println("W_S_takeoff_climb6, T_W_takeoff_climb6 for this intersection point");


    # 4. Climb 6 與 Service Ceiling 交點 (兩條曲線相交)
    diff_climb6_ceiling = abs.(balked_OEI_climb - TbW_ceiling);
    idx_climb6_ceiling = argmin(diff_climb6_ceiling);
    W_S_climb6_ceiling = wing_loading[idx_climb6_ceiling];
    T_W_climb6_ceiling = balked_OEI_climb[idx_climb6_ceiling] ; # 或 TbW_ceiling[idx_climb6_ceiling]
    push!(critical_points, (W_S_climb6_ceiling, T_W_climb6_ceiling));
    push!(point_names, "Climb6-Ceiling");
    println("\n4. Climb 6 與 Service Ceiling 交點:");
    println("   W/S = $(round(W_S_climb6_ceiling, digits=1)) N/m²");
    println("   T/W = $(round(T_W_climb6_ceiling, digits=3))");
	println("W_S_climb6_ceiling, T_W_climb6_ceiling for this intersection point");
    
    # 5. Landing 與 Climb 6 交點 (垂直線與曲線)
    W_S_landing = WbS_landing_max;  # 著陸限制翼載
    idx_landing_climb6 = argmin(abs.(wing_loading .- W_S_landing));
    T_W_landing_climb6 = balked_OEI_climb[idx_landing_climb6];
    push!(critical_points, (W_S_landing, T_W_landing_climb6));
    push!(point_names, "Landing-Climb6");
    println("\n5. Landing 與 Climb 6 交點:");
    println("   W/S = $(round(W_S_landing, digits=1)) N/m²");
    println("   T/W = $(round(T_W_landing_climb6, digits=3))");
    println("W_S_landing, T_W_landing_climb6 for this intersection point");

    
end

global W_S_takeoff_climb6
global T_W_takeoff_climb6
global W_S_stall
global T_W_stall_takeoff
global W_S_landing
global T_W_landing_takeoff
global W_S_landing
global T_W_landing_climb6


In [ ]:
#===========================================#
# Change The Design Point Here (and check the design point) Exported
#============================================#
design_W_S = W_S_stall
design_T_W = T_W_stall_takeoff

# design_W_S = W_S_takeoff_climb6 
# design_T_W = T_W_takeoff_climb6

# design_W_S = W_S_landing
# design_T_W = T_W_landing_takeoff

# design_W_S = W_S_landing
# design_T_W = T_W_landing_climb6

# design_W_S = W_S_climb6_ceiling
# design_T_W = T_W_climb6_ceiling
#--------------------------------------------------------# 
begin
	p_match = plot(
		xlabel = "Wing Loading (W/S), N/m²", 
		ylabel = "Thrust Loading (T/W)", 
		title = "Matching Chart",
		legend = :topright,
		legendfontsize = 7,
		grid = true,
        minorgrid = true,
        size = (1000, 750),  # 設定圖表大小
        dpi = 300,
        ylims = (0, 0.6),
		xlims = (850, 8000))
	   # ✅ Measures.AbsoluteLength
	# Lines
	plot!(stalls, TbWs, label = "Stall", color = :gray50)
	plot!(wing_loading, TbW_takeoff, label = "Takeoff",color = :blue)
	plot!(WbS_landing, TbWs, label = "Landing", color = :teal)
	plot!(wing_loading, takeoff_climbs, label = "Takeoff Climb",color = :steelblue)
	plot!(wing_loading, trans_climb, label = "Transition Climb OEI",color = :cyan)
	plot!(wing_loading, second_climb, label = "Second Climb OEI",color = :red)
	plot!(wing_loading, enroute_climb, label = "Enroute Climb OEI", color = :orange)
	plot!(wing_loading, balked_AEO_climb, label = "Balked Landing Climb AEO", color = :violet)
	plot!(wing_loading, balked_OEI_climb, label = "Balked Landing Climb OEI", color = :purple)
	plot!(wing_loading, TbW_cruise, label = "Cruise",color = :green)
	plot!(wing_loading, T_W_loiter, label = "Loiter", color = :indigo)
	plot!(wing_loading, TbW_ceiling, label = "Service Ceiling",color = :magenta)
	plot!(wing_loading, TbW_turn, label = "Turning",color = :brown)
	plot!(wing_loading, TbW_takeoff_accel, label = "Takeoff Acceleration", color = :darkred)
	plot!(wing_loading, TbW_cruise_accel, label = "Cruise Acceleration", color = :black)
	
	W_S_landing = WbS_landing_max # 4121 N/m²
    
	#Find Design point


    #design_W_S
    #design_T_W
    scatter!([design_W_S], [design_T_W], label = "", markersize=10, color=:red, marker=:circle)
    println("\n" * "="^60);
 	display(p_match);

    
end


In [ ]:
#================================================#
# Update Aircraft Parameters 
#================================================#

begin
    println("\n" * "="^60)
    println("AIRCRAFT PARAMETERS FROM DESIGN POINT")
    println("="^60)
    
    # cal fundamental parameters
    W_S_kgpm2 = design_W_S / g0
    S_ref = WTO_final * g0 / design_W_S  # 初始机翼面积（基于猜测重量）
    T_total = design_T_W * WTO_final * g0  # 总推力 (N)
    T_per_engine = T_total / 2 / 1000  # 每台发动机推力 (kN)
    
    println("Design Point Values:")
    println("  Wing Loading: $(round(W_S_kgpm2, digits=1)) kg/m² ($(round(design_W_S, digits=1)) N/m²)")
    println("  Thrust-to-Weight: $(round(design_T_W, digits=3))")
    println("\nSizing (based on WTO_final = $(WTO_final) kg):")
    println("  Wing Area: $(round(S_ref, digits=1)) m²")
    println("  Total Thrust Required: $(round(T_total/1000, digits=1)) kN")
    println("  Thrust per Engine: $(round(T_per_engine, digits=1)) kN")
    
    # check wing limitation（Code B: <24m）
    span = sqrt(AR * S_ref)
    println("\nWing Span: $(round(span, digits=1)) m")
    if span < 24
        println("✅ Wingspan meets Code B requirement (<24m)")
    else
        println("❌ Wingspan exceeds Code B requirement!")
        println("   Need to reduce span or increase AR")
    end
    
    # 检查 TOP 是否合理
    TOP_check = (design_T_W * 37.5) / (CLmax_takeoff * sigma_sl)  # 反算 TOP
    TOP_check_eng = TOFL_required * 3.28084 * sigma_sl * CLmax_takeoff *design_T_W / 37.5
    println("\nTakeoff Parameter (TOP): $(round(TOP_check_eng, digits=1))")
    if TOP_check_eng >= 200 && TOP_check_eng <= 350
        println("✅ TOP is in reasonable range (200-350)")
    else
        println("⚠️ TOP is outside typical range - check takeoff performance")
    end
    
    println("="^60)
end



In [ ]:
##################################################################
# For checking
######################################################################
# W_S_target = W_S_stall
# W_S_kg_target = T_W_stall_takeoff
# W_S_target = W_S_takeoff_climb6
# W_S_kg_target = T_W_takeoff_climb6


# 目标翼载（满足翼展 24.0 m）
 W_S_target = W_S_stall
 W_S_kg_target = T_W_stall_takeoff

# 在约束曲线上找到对应的 T/W
idx = argmin(abs.(wing_loading .- W_S_target))
T_W_target = max(
    TbW_takeoff[idx],
    second_climb[idx],
    TbW_cruise[idx],
    TbW_ceiling[idx],
    T_W_loiter[idx],
    balked_OEI_climb[idx]
)

# 重新计算参数
S_target = WTO_final * g / W_S_target
span_target = sqrt(AR * S_target)
T_total_target = T_W_target * WTO_final * g / 1000
TOP_target = 1800 * 3.28084 * sigma_sl * CLmax_takeoff * T_W_target / 37.5

println("="^60)
println("FINAL ADJUSTED DESIGN")
println("="^60)
println("Wing Loading: $(round(W_S_target)) N/m² ($(round(W_S_kg_target, digits=1)) kg/m²)")
println("Thrust-to-Weight: $(round(T_W_target, digits=3))")
println("Wing Area: $(round(S_target, digits=1)) m²")
println("Wing Span: $(round(span_target, digits=1)) m")
if span_target <= 24
    println("  ✅ Meets Code B requirement")
else
    println("  ❌ Still exceeds by $(round(span_target-24, digits=2)) m")
end
println("Total Thrust: $(round(T_total_target, digits=1)) kN")
println("Thrust per Engine: $(round(T_total_target/2, digits=1)) kN")
println("Takeoff Parameter (TOP): $(round(TOP_target, digits=1))")
println("="^60)


In [ ]:
##############################################
#For checking
#################################################

# Cal the thrust lapse for 35000 ft by python model
h_35000ft = 35000 * 0.3048  # in m
thrust_coeff = calc_thrust_lapse(h_35000ft, 250.0, 0.0, 1.0)
println("Thrust coeff= $(round(thrust_coeff, digits=3))")



h_cruise = 35000 * 0.3048
atm_data = atmosphere.compute_constants(h_cruise, 0.0)  # ← 改成 atmosphere

rho_py_cruise = atm_data["rho"]
v_sound_py = atm_data["v_sound"]
println("Rho = $(round(rho_py_cruise, digits=3)) kg/m³")
println("speed = $(round(v_sound_py, digits=1)) m/s")

TOFL_required = 1800  # m
n_engines = 2
TOP_value_py = calc_TOP(TOFL_required, n_engines)
println("TOP required = $(round(TOP_value_py, digits=1)) lbf/ft²")

# Print the result cal by manual
println("\nDensity (kg/m³):")
println("  Manual: $(round(rho_cruise, digits=3))")
println("  Diff: $(round(abs(rho_py_cruise - rho_cruise)/rho_py_cruise*100, digits=2))%")
    
println("\nSpeed of sound (m/s):")
println("  Manual: $(round(a_cruise, digits=1))")
println("  Diff: $(round(abs(v_sound_py - a_cruise)/v_sound_py*100, digits=2))%")

#-----------------------------------------------------------------------------
# Check the Takeoff Time
aero_data = Dict(
    "AR_wing" => AR,
    "e_clean" => e_cruise,
    "CD0_clean" => CD0_cruise
)

ttc = calculate_time_to_climb(WTO_final, S_ref, T_total_target*1000, aero_data, 35000)
println("Time to Climb required = $(round(ttc, digits=2)) min")

println("="^60)
println("TIME TO CLIMB CALCULATION")
println("="^60)
println("MTOW: $(round(WTO_final, digits=1)) kg")
println("Wing Area: $(round(S_ref, digits=1)) m²")
println("Total Thrust: $(round(T_total/1000, digits=1)) kN")
println("Time to Climb to 35,000 ft: $(round(ttc, digits=2)) minutes")

if ttc <= 18
    println("✅ Meets RFP requirement (≤18 min)")
else
    println("❌ Does NOT meet RFP requirement!")
end
    println("="^60)


In [ ]:
#############################################################
## Bisection Method (From tutorial)
################################################################

function compute_WTO_residual(WTO, W_payload, W_crew, Wf_WTO, A, B)
	We_WTO = empty_weight_raymer(WTO, A, B) #kg
	WTO_calc = maximum_takeoff_weight(W_payload, W_crew, Wf_WTO, We_WTO)
	residual = WTO - WTO_calc
	return residual #kg
end


function compute_WTO_via_bisection_method(
		f::Function, a::Number, b::Number,
		# Input arguments
		W_payload, W_crew, Wf_WTO, A, B;
		# Default arguments
		num_iters = 40, # number of iterations
		tol = 1e-12 	# convergence tolerance
	)

	WTO_list_kg, residual_list_kg = [], []
	
    fa = f(a, W_payload, W_crew, Wf_WTO, A, B)
    fa*f(b, W_payload, W_crew, Wf_WTO, A, B) <= 0 || error("No real root in [a,b]")
    i = 0
    local c

	for i in 1:num_iters
		
        c = (a+b)/2
        fc = f(c, W_payload, W_crew, Wf_WTO, A, B)
		push!(WTO_list_kg, c)
		push!(residual_list_kg, abs(fc))

		# Conditional
		if abs(fc) < tol
			break  # break loop if abs(residual) is less than tolerance
        elseif fa*fc > 0
            a = c  # Root is in the right half of [a,b].
            fa = fc
        else
            b = c  # Root is in the left half of [a,b].
        end
    end
    return WTO_list_kg, residual_list_kg
end


begin
	max_iter_bisect = 40
	WTO_guess1 = 10800.0*lb_to_kg  # in lb, initial guess 1
	WTO_guess2 = 100000.0*lb_to_kg # in lb, initial guess 2
end;

println("="^60)
    println("PARAMETERS (All units in kg)")
    println("="^60)
    println("Payload:      $(W_payload) kg")
    println("Crew:         $(W_crew) kg")
    println("Total load:   $(W_pl_crew) kg")
    println("Fuel fraction: $(Wf_WTO)")
    println("Raymer A:     $A_raymer")
    println("Raymer B:     $B_raymer")
    println("Bisection bounds:")
    println("  Lower: $(round(WTO_guess1, digits=1)) kg ($(round(WTO_guess1/lb_to_kg, digits=1)) lb)")
    println("  Upper: $(round(WTO_guess2, digits=1)) kg ($(round(WTO_guess2/lb_to_kg, digits=1)) lb)")



WTO_list2, residual_list = compute_WTO_via_bisection_method(
							compute_WTO_residual, WTO_guess1, WTO_guess2,
							W_payload, W_crew, Wf_WTO, A_raymer, B_raymer,
							num_iters = 40, tol=1e-3
);

WTO_result2_kg = WTO_list2[end] # in kg
WTO_result2_lb = WTO_result2_kg / lb_to_kg #in lb

# Display results
println("\n" * "="^60)
println("BISECTION METHOD RESULTS")
println("="^60)
println("WTO convergence history (kg):")
for (i, wto) in enumerate(WTO_list2)
    println("  Iter $i: $(round(wto, digits=1)) kg")
end

println("\nResidual history:")
for (i, res) in enumerate(residual_list)
    println("  Iter $i: $(round(res, digits=6))")
end


println("\n" * "-"^60)
println("Final Results:")
println("  MTOW = $(round(WTO_result2_lb, digits=1)) lb = $(round(WTO_result2_kg, digits=1)) kg")
println("  Final residual = $(round(residual_list[end], digits=8))")
println("  Iterations = $(length(WTO_list2))")

#---------------------------------------------------------------#
#Compare the fixed-point iteration with bisection method result
#---------------------------------------------------------------

println("\n" * "-"^60)
println("Compare with the result obtained from Fixed-point Iteration:")
println("  Final MTOW (fixed-point): $(round(WTO_list[end], digits=1)) kg")
println("  Difference: $(round(abs(WTO_list[end]-WTO_result2_kg)/WTO_list[end]*100, digits=4)) %")
println("  Empty weight fraction(fixed-point): $(round(We_WTO_final, digits=4))")
println("  Empty weight(fixed-point): $(round(We_final, digits=1)) kg = $(round(We_final * kg_to_lb, digits=1)) lb")

#Plotting
begin
	# plot3 = plot(title="Bisection method",
	# 			 grid=false, showaxis=false, bottom_margin = -140Plots.px)
	plot4 = plot(WTO_list2, 
		label = "", 
		ylabel = "Takeoff weight (lb)", 
		xlabel = "Iterations"
	)
	plot5 = plot(residual_list,
		label = "", 
		ylabel = "Residual", 
		yscale = :log10, 
		xlabel = "Iterations"
	)
	#plot(plot3)
	plot(plot4, plot5, layout = (2,1))
	#display(plot3)
	display(plot4)
	display(plot5)
end



In [ ]:
#Final result
W_empty = empty_weight_raymer(WTO_final, A_raymer, B_raymer)

W_residual = WTO_final - (We_final + Wf_final + W_payload + W_crew)


In [ ]:
#===================================================================#
# Estimate if we use % of composite material
#==================================================================#

composite_saving = 0.2 # 0.05 --> 5%

function compute_WTO_with_composites(
		# Input arguments
		WTO_guess, W_payload, W_crew, Wf_WTO, A_raymer, B_raymer, composite_saving,
		# Default arguments
		num_iters = 40, # number of iterations
		tol = 1e-12 	# convergence tolerance
	)

	# Initial value of takeoff weight from guessing
	WTO = Float64(WTO_guess)

	# Array of MTOW iterative values
	WTO_list = Float64[WTO]

	# Array of errors over iterations of size num_iters, initially infinite
	error_list = [ Inf; zeros(num_iters) ]
		
	# Iterative loop
	for i in 1:num_iters
		
		# Calculate empty weight fraction and consider composite saving
		We_WTO = (1 - composite_saving) * empty_weight_raymer(WTO, A_raymer, B_raymer)

		# Calculate new MTOW with the calculated empty weight fraction
		WTO_new = maximum_takeoff_weight(W_payload, W_crew, Wf_WTO, We_WTO)

		# Evaluate relative error
		error = abs((WTO_new - WTO)/WTO)

		# Append MTOW_new to MTOW_list
		push!(WTO_list, WTO_new)
        push!(error_list, error)
		WTO_lb_new = WTO_new * kg_to_lb
        println(rpad(i, 5), "| ", 
                rpad(round(WTO_new, digits=1), 15), "| ", 
                rpad(round(WTO_lb_new, digits=1), 15), "| ", 
                rpad(round(We_WTO, digits=4), 8), "| ", 
                round(error*100, digits=6))

				
		# Assign error to error_list
		error_list[i] = error

		# Conditional
		if error < tol
			break 				# break loop
		else
			WTO = WTO_new 	# assign new value
		end
	end

	# Return arrays of MTOW and error
	return WTO_list, error_list, We_WTO

end
println("="^60)
println("COMPOSITE MATERIAL ANALYSIS ($(composite_saving*100)% weight saving)")
println("="^60)
println("Iter |    MTOW (kg)    |    MTOW (lb)    |  We/WTO  |  Error (%)")
println("-"^70)

WTO_list3, error_list3 = compute_WTO_with_composites(W_payload+W_crew, W_payload, W_crew, Wf_WTO, A_raymer, B_raymer, composite_saving)

# Takeoff gross weight
WTO_result3 = WTO_list3[end]

total_weight_saving = (WTO_result3-WTO_final)/WTO_final * 100 # %

println("-"^70)
println("   Final MTOW: $(round(WTO_result3, digits=1)) kg")
println("   Empty Weight Saving: $(round(-total_weight_saving, digits=8))%")
println("Empty weight: $(round(We_WTO))")


In [ ]:
#Cal the new result

# 重新计算参数
S_target_composite = WTO_result3 * g / W_S_target
span_target_composite = sqrt(AR * S_target_composite)
T_total_target_composite = T_W_target * WTO_result3 * g / 1000
TOP_target_composite = 1800 * 3.28084 * sigma_sl * CLmax_takeoff * T_W_target / 37.5

println("="^60)
println("FINAL ADJUSTED DESIGN")
println("="^60)
println("Wing Loading: $(round(W_S_target)) N/m² ($(round(W_S_kg_target, digits=1)) kg/m²)")
println("Thrust-to-Weight: $(round(T_W_target, digits=3))")
println("Wing Area: $(round(S_target_composite, digits=1)) m²")
println("Wing Span: $(round(span_target_composite, digits=1)) m")
if span_target_composite <= 24
    println("  ✅ Meets Code B requirement")
else
    println("  ❌ Still exceeds by $(round(span_target_composite-24, digits=2)) m")
end
println("Total Thrust: $(round(T_total_target_composite, digits=1)) kN")
println("Thrust per Engine: $(round(T_total_target_composite/2, digits=1)) kN")
println("Takeoff Parameter (TOP): $(round(TOP_target_composite, digits=1))")
println("="^60)


#-----------------------------------------------------------------------------
# Check the Takeoff Time
aero_data = Dict(
    "AR_wing" => AR,
    "e_clean" => e_cruise,
    "CD0_clean" => CD0_cruise
)

ttc_composite = calculate_time_to_climb(WTO_result3, S_target_composite, T_total_target*1000, aero_data, 35000)
println("Time to Climb required = $(round(ttc, digits=2)) min")

println("="^60)
println("TIME TO CLIMB CALCULATION")
println("="^60)
println("MTOW: $(round(WTO_result3, digits=1)) kg")
println("Wing Area: $(round(S_target_composite, digits=1)) m²")
println("Total Thrust: $(round(T_total_target_composite, digits=1)) kN")
println("Time to Climb to 35,000 ft: $(round(ttc_composite, digits=2)) minutes")

if ttc_composite <= 18
    println("✅ Meets RFP requirement (≤18 min)")
else
    println("❌ Does NOT meet RFP requirement!")
end
    println("="^60)



# Export Setting


In [ ]:
#================================================#
# export result to TXT 檔案
#================================================#

begin
    
    
    # create and export folder
    output_dir = "design_output_$(Dates.format(now(), "yyyy-mm-dd_HH-MM"))"
    mkpath(output_dir)
    
    # create TXT file
    txt_file = joinpath(output_dir, "complete_design_data.txt")
    open(txt_file, "w") do io
    
        println(io, "="^60)
        println(io, "COMPLETE AIRCRAFT DESIGN DATA")
        println(io, "Generated: $(Dates.now())")
        println(io, "="^60)
        
        # 1. basic parameter
        println(io, "\n1. MISSION PARAMETERS")
        println(io, "-"^40)
        
        if isdefined(Main, :W_payload)  # 改用 isdefined(Main, :變數名)
            println(io, "Payload weight: $(W_payload) kg")
        end
        if isdefined(Main, :W_crew)
            println(io, "Crew weight: $(W_crew) kg")
        end
        if isdefined(Main, :R)
            println(io, "Range: $(R/1000) km")
        end
        if isdefined(Main, :M)
            println(io, "Cruise Mach: $(M)")
        end
        
        # 2. aereodynamic parameter------------------------------------------------
        println(io, "\n2. AERODYNAMIC PARAMETERS")
        println(io, "-"^40)
        
        if isdefined(Main, :CLmax_takeoff)
            println(io, "CLmax_takeoff = $(CLmax_takeoff)")
        end
        if isdefined(Main, :CLmax_landing)
            println(io, "CLmax_landing = $(CLmax_landing)")
        end
        if isdefined(Main, :CD0)
            println(io, "CD0 = $(CD0)")
        end
        if isdefined(Main, :CD0_cruise)
            println(io, "CD0_cruise = $(CD0_cruise)")
        end
        if isdefined(Main, :AR)
            println(io, "AR = $(AR)")
        end
        if isdefined(Main, :e_cruise)
            println(io, "e_cruise = $(e_cruise)")
        end
        if isdefined(Main, :V_stall)
            println(io, "V_stall = $(V_stall)")
        end
        
        # 3. mission analysis result---------------------------------------------------
        println(io, "\n3. MISSION ANALYSIS RESULTS")
        println(io, "-"^40)
        
        if isdefined(Main, :beta_landing1)
            println(io, "Beta (landing): $(round(beta_landing1, digits=4))")
            println(io, "One-way fuel burn: $(round((1-beta_landing1)*100, digits=1))%")
        end
        
        if isdefined(Main, :final_beta)
            println(io, "Final beta: $(round(final_beta, digits=4))")
            println(io, "Mission fuel fraction: $(round((1-final_beta)*100, digits=1))%")
        end

        if isdefined(Main, :Wf_WTO)
            println(io, "Total fuel fraction(with 6% trapped): $(round(Wf_WTO*100, digits=1))%")
        end


        # 4. design point selected--------------------------------------------------------
        println(io, "\n4. DESIGN POINT")
        println(io, "-"^40)
        
        if isdefined(Main, :design_W_S)
            println(io, "Wing Loading: $(design_W_S) N/m² ($(round(design_W_S/9.81, digits=1)) kg/m²)")
        end
        if isdefined(Main, :design_T_W)
            println(io, "Thrust-to-Weight: $(design_T_W)")
        end
        
        # 5. weight final-----------------------------------------------------------------
        println(io, "\n5. WEIGHTS")
        println(io, "-"^40)
        
        if isdefined(Main, :WTO_final)
            println(io, "MTOW: $(round(WTO_final, digits=1)) kg")
        end
        if isdefined(Main, :fuel_weight)
            println(io, "Fuel: $(round(fuel_weight, digits=1)) kg")
        end
        if isdefined(Main, :payload)
            println(io, "Payload: $(round(payload, digits=1)) kg")
        end

        
        # 6. The geometry result------------------------------------------------
        println(io, "\n6. GEOMETRY")
        println(io, "-"^40)
        
        if isdefined(Main, :S_ref)
            println(io, "Wing Area: $(round(S_ref, digits=1)) m²")
        end
        if isdefined(Main, :span)
            println(io, "Wing Span: $(round(span, digits=2)) m")
        end
        
        # 7. TTC---------------------------------------------------------------------------
        println(io, "\n7. PERFORMANCE")
        println(io, "-"^40)
        
        if isdefined(Main, :TOP_value_py)
            println(io, "TOP required = $(round(TOP_value_py, digits=1)) lbf/ft²")
        end
        if isdefined(Main, :ttc)
            println(io, "Time to Climb to 35,000 ft: $(round(ttc, digits=2)) minutes")
        end
        
        # 8. all intersection point-------------------------------------------------------------
        println(io, "\n8. INTERSECTION POINTS")
        println(io, "-"^40)
        
        if isdefined(Main, :critical_points) && isdefined(Main, :point_names)
            for (i, (ws, tw)) in enumerate(critical_points)
                name = i <= length(point_names) ? point_names[i] : "Point $i"
                println(io, "$name: W/S=$(round(ws, digits=1)) N/m², T/W=$(round(tw, digits=3))")
            end
        end
        
        # 9. Empty weight ratio-----------------------------------------------------------------
        println(io, "\n9. EMPTY WEIGHT MODELS (can ignore the roskam one)")
        println(io, "-"^40)
        
        #Raymer
        if isdefined(Main, :We_WTO_final) && isdefined(Main, :We_final)
            println(io, "Raymer: $(round(We_final, digits=1)) kg ($(round(We_WTO_final*100, digits=1))%)")
        end
        #Roskam
        # if isdefined(Main, :We_WTO_roskam) && isdefined(Main, :We_roskam)
        #     println(io, "Roskam: $(round(We_roskam, digits=1)) kg ($(round(We_WTO_roskam*100, digits=1))%)")
        # end
        
        #10. Use of Composite material
        println(io, "\n10. COMPOSITE MATERIAL ANALYSIS")
        println(io, "-"^40)

        if isdefined(Main, :composite_saving)
        println(io, "$(composite_saving*100)% weight saving")
        end

        if isdefined(Main, :WTO_result3) 
        println(io, "Final MTOW: $(round(WTO_result3, digits=1)) kg")
        end

        if isdefined(Main, :total_weight_saving)
        println(io, "Empty Weight Saving: $(round(-total_weight_saving, digits=8))%")
        end

        #11. Bisection Method
        println(io, "\n11. BISECTION METHOD RESULTS")
        println(io, "-"^40)

        if isdefined(Main, :WTO_result2_kg)
        println(io, "Final MTOW (Bisection Method): $(round(WTO_result2_kg, digits=1))%")
        end

        if isdefined(Main, :difference_fixedpt_bisection)
        println(io, "Difference: $(round(abs(WTO_list[end]-WTO_result2_kg)/WTO_list[end]*100, digits=4)) %")
        end
        println(io, "\n" * "="^60)
    end
    
    println("✅ data saved in: $txt_file")
    
    # 保存圖表
    if isdefined(Main, :p_match)
        try
            savefig(p_match, joinpath(output_dir, "matching_chart.pdf"))
            savefig(p_match, joinpath(output_dir, "matching_chart.png"))
            println("✅ matching chart saved")
        catch e
            println("⚠️Cannot save the matching chart: $e")
        end
    else
        println("⚠️ Variable p_match not exist，skipped matching chart")
    end
    
    if isdefined(Main, :p_final)
        try
            savefig(p_final, joinpath(output_dir, "iteration_convergence.pdf"))
            savefig(p_final, joinpath(output_dir, "iteration_convergence.png"))
            println("✅ Iteration graph saved")
        catch e
            println("⚠️ Fail to save interation graph: $e")
        end
    else
        println("⚠️ 變數 p_final 不存在，跳過收斂圖")
    end
    
    if isdefined(Main, :p_design_point)
        try
            savefig(p_design_point, joinpath(output_dir, "design_point.pdf"))
            savefig(p_design_point, joinpath(output_dir, "design_point.png"))
            println("✅ 設計點圖已保存")
        catch e
            println("⚠️ 無法保存設計點圖: $e")
        end
    else
        println("⚠️ 變數 p_design_point 不存在，跳過設計點圖")
    end

    if isdefined(Main, :p_raymer)
        try
            savefig(p_raymer, joinpath(output_dir, "Model_comparison.pdf"))
            savefig(p_raymer, joinpath(output_dir, "Model_comparison.png"))
            println("✅ 設計點圖已保存")
        catch e
            println("⚠️ 無法保存設計點圖: $e")
        end
    else
        println("⚠️ 變數 p_design_point 不存在，跳過設計點圖")
    end

    if isdefined(Main, :p_mission)
        try
            savefig(p_mission, joinpath(output_dir, "Mission_Profile.pdf"))
            savefig(p_mission, joinpath(output_dir, "Mission_Profile.png"))
            println("✅ 設計點圖已保存")
        catch e
            println("⚠️ 無法保存設計點圖: $e")
        end
    else
        println("⚠️ 變數 p_mission 不存在，跳過設計點圖")
    end
    
    println("\n✅ all result saved into: $output_dir")
    println("   data: complete_design_data.txt")
    println("="^60)


end
